# nexus-lite v0.2.2+ kernel primitives on Colab

Runnable tour of the three primitive areas added in v0.2.2+: (1) memory tiers + Bus v1 semantics, (2) the `NEXUS_SENSITIVITY=phi` gate with `stm/1` redaction, (3) zero-preinstall remote compute (`pack_self_contained_script` / `run_on_colab`) over a local loopback tunnel. No GPU, no API keys.

## 1. Install

In [ ]:
!pip install -q git+https://github.com/choilab-jefferson/nexus.lite.git

## 2. Memory tiers: STM, MTM, LTM
Everything lives in an isolated scratch workspace (`NEXUS_LITE_DIR`). MTM flow checkpoints are bus messages of `kind='checkpoint'`; the bus is the MTM transport.

In [ ]:
import os, tempfile
os.environ.pop('NEXUS_SENSITIVITY', None)
os.environ['NEXUS_LITE_DIR'] = tempfile.mkdtemp(prefix='nexus-v022-')
from nexus_lite import _db
_db.reset_init()
from nexus_lite.memory import stm, mtm, ltm

# STM: file-backed stm/1 nodes
node = stm.stm_write(kind='task', status='running', summary='segment lung nodules', agent='worker-1')
print('stm_read   :', stm.stm_read(node['id'])['summary'])
print('stm_pack   :', stm.stm_pack()['schema'], 'nodes =', stm.stm_pack()['node_count'])

# MTM: flow checkpoint + state carried over the bus
cp = mtm.bus_send('orchestrator', 'step 2/5 done: features extracted', sender='worker-1', kind='checkpoint')
print('mtm state  :', [m['summary'] for m in mtm.bus_read('orchestrator') if m['kind'] == 'checkpoint'])

# LTM: markdown wiki + FTS5 search
ltm.wiki_write('notes/radiomics', '# Radiomics\nIBSI first-order features need discretisation.')
print('ltm search :', [h['slug'] for h in ltm.wiki_search('discretisation')])

## 3. Bus v1: monotonic `seq`, ack cursors, offline redelivery, prune

In [ ]:
m1 = mtm.bus_send('bob', 'first', sender='al')
m2 = mtm.bus_send('bob', 'second', sender='al')
print('monotonic seq:', m1['seq'], '<', m2['seq'])

# bob was offline when the messages were sent; registering redelivers them
print('redelivered  :', mtm.bus_register('bob')['redelivered'] == [m1['id'], m2['id']])

# ack moves the cursor (never backwards)
print('bus_ack      :', mtm.bus_ack('bob', m1['seq']))
with _db.connect() as con:
    print('bus_cursors  :', [(r['target'], r['acked_seq']) for r in con.execute('SELECT target, acked_seq FROM bus_cursors')])
print('unacked      :', [m['summary'] for m in mtm.bus_read('bob', unacked=True)])

# prune deletes only acked messages (seq 1); the unacked one survives
print('bus_prune    :', mtm.bus_prune(0), 'deleted')
print('unacked left :', [m['summary'] for m in mtm.bus_read('bob', unacked=True)])

## 4. PHI sensitivity gate
With `NEXUS_SENSITIVITY=phi` only local runners (local Ollama, shell, python) may be dispatched; cloud CLIs are refused. `stm/1` summaries are redacted in the packed brief.

In [ ]:
from nexus_lite.phi import phi_gate_error
os.environ['NEXUS_SENSITIVITY'] = 'phi'
for cmd in ['claude', 'agy -p', 'ollama launch claude', 'ollama run llama3.2']:
    err = phi_gate_error(cmd)
    print(f'{cmd!r:28} ->', 'BLOCKED' if err else 'allowed (local)')

stm.stm_write(kind='task', summary='patient John Doe MRN 12345')
brief = stm.stm_pack(phi=True)
print('sensitivity:', brief['sensitivity'], '| headline:', brief['headline'])
print('summaries  :', [n['summary'] for n in brief['nodes']])
assert all(n['summary'] == '' for n in brief['nodes'])
os.environ.pop('NEXUS_SENSITIVITY')

## 5. Zero-preinstall remote compute
`pack_self_contained_script` builds one self-extracting `.py` that carries `nexus_lite`; `run_on_colab` ships it to a runtime over `POST /exec`. Here the 'remote' is a loopback tunnel server in this same runtime, so it runs with no external setup.

In [ ]:
import nexus_lite.compute as compute
packed = compute.pack_self_contained_script(python='RESULT = 6 * 7')
print('packed script:', len(packed[0]), 'bytes')

from nexus_lite.colab import tunnel as colab_tunnel
info = colab_tunnel.start_tunnel(port=0, tunnel=False)
try:
    tok = colab_tunnel.token_file_path(info['port']).read_text().strip()
    res = compute.run_on_colab(python='RESULT = {"answer": 6 * 7}', url=info['local_url'], token=tok)
    print('run_on_colab result:', res['result'])
finally:
    colab_tunnel.stop_tunnel()